# OptiCell Stage 2 — CTC **TRA / DET / LNK** scoring (Colab)

Scores OptiCell tracks against CTC ground truth with [`traccuracy`](https://github.com/live-image-tracking-tools/traccuracy).

**Per sequence:** download CTC → Stage-2 (threshold+tracking) → export RES → `CTCMetrics` (TRA/DET/LNK).

**Fix (latest `main`):** `export_ctc_res` only writes track IDs that exist as pixels in each mask, so traccuracy no longer raises `Missing IDs in masks`.

Measured numbers only. Cite CTC if publishing.

## 0. Setup — pull latest

In [ ]:
import os, sys, json, zipfile, urllib.request, shutil, subprocess
from pathlib import Path

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/opticell_tra')
else:
    BASE = Path('/content/opticell_tra')
BASE.mkdir(parents=True, exist_ok=True)
print('BASE', BASE)

In [ ]:
REPO = Path('/content/Virelion-OptiCell')
if not REPO.exists():
    !git clone https://github.com/Virelion-Biotech/Virelion-OptiCell.git
else:
    !git -C /content/Virelion-OptiCell fetch origin main
    !git -C /content/Virelion-OptiCell reset --hard origin/main

%cd /content/Virelion-OptiCell
!pip install -e . -q
!pip install -q traccuracy tifffile

from traccuracy import run_metrics
from traccuracy.loaders import load_ctc_data
from traccuracy.matchers import CTCMatcher
from traccuracy.metrics import CTCMetrics

# ensure fixed exporter is on path
import importlib
sys.path.insert(0, str(REPO / 'scripts'))
import export_ctc_res as _exp
importlib.reload(_exp)
from export_ctc_res import export_ctc_res

print('ready', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())

## 1. Helpers

In [ ]:
DATASETS = {
    'Fluo-N2DH-GOWT1': {
        'url': 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-GOWT1.zip',
        'mb': 53,
        'sequences': ['01', '02'],
    },
    'Fluo-N2DH-SIM+': {
        'url': 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-SIM+.zip',
        'mb': 91,
        'sequences': ['01', '02'],
    },
    'Fluo-N2DL-HeLa': {
        'url': 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DL-HeLa.zip',
        'mb': 182,
        'sequences': ['01', '02'],
    },
}

CTC_ROOT = BASE / 'ctc'
CTC_ROOT.mkdir(parents=True, exist_ok=True)

def download_and_extract(name: str) -> Path:
    meta = DATASETS[name]
    dest = CTC_ROOT / name
    zpath = CTC_ROOT / f'{name}.zip'
    if dest.exists() and any(dest.iterdir()):
        print('[skip download]', dest)
        return dest
    print(f'[download] {name} (~{meta["mb"]} MB)')
    urllib.request.urlretrieve(meta['url'], zpath)
    with zipfile.ZipFile(zpath, 'r') as zf:
        zf.extractall(CTC_ROOT)
    if not dest.exists():
        raise FileNotFoundError(dest)
    return dest

def find_seq_and_gt(root: Path, sequence: str):
    seq = root / sequence
    gt_tra = root / f'{sequence}_GT' / 'TRA'
    man = gt_tra / 'man_track.txt'
    if not seq.is_dir() or not man.is_file():
        raise FileNotFoundError((seq, gt_tra, man))
    return seq, gt_tra, man

def run_stage2(seq_dir: Path, out_dir: Path, max_frames: int = 0,
               track_max_distance: float = 50.0, track_max_gap: int = 1):
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)
    cmd = [
        sys.executable, str(REPO / 'scripts' / 'run_killer_workflow.py'),
        str(seq_dir), '-o', str(out_dir),
        '--backend', 'threshold', '--enable-tracking',
        '--track-max-distance', str(track_max_distance),
        '--track-max-gap', str(track_max_gap),
    ]
    if max_frames > 0:
        cmd += ['--max-images', str(max_frames)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stdout or '')[-2500:])
        print((r.stderr or '')[-2500:])
        raise RuntimeError(f'stage2 failed {r.returncode}')
    if not (out_dir / 'tracks.csv').is_file():
        raise RuntimeError('tracks.csv missing')
    return out_dir

def score_tra(gt_tra: Path, man_track: Path, res_dir: Path, name: str):
    gt = load_ctc_data(str(gt_tra), str(man_track), name=f'{name}_GT')
    pred = load_ctc_data(str(res_dir), str(res_dir / 'res_track.txt'), name=f'{name}_RES')
    results, _ = run_metrics(
        gt_data=gt,
        pred_data=pred,
        matcher=CTCMatcher(),
        metrics=[CTCMetrics()],
    )
    block = results[0] if isinstance(results, list) else results
    if hasattr(block, 'results'):
        metrics = dict(block.results)
    elif isinstance(block, dict) and 'results' in block:
        metrics = dict(block['results'])
    else:
        metrics = dict(block)
    return metrics

def run_one(dataset: str, sequence: str, max_frames: int = 0,
            track_max_distance: float = 50.0, reuse_stage2: bool = False):
    root = download_and_extract(dataset)
    seq_dir, gt_tra, man = find_seq_and_gt(root, sequence)
    tag = f'{dataset}_{sequence}'
    stage2_dir = BASE / 'stage2' / tag
    res_dir = BASE / 'res' / tag

    if reuse_stage2 and (stage2_dir / 'tracks.csv').is_file():
        print(f'=== {tag} reuse Stage-2 ===', stage2_dir)
    else:
        print(f'=== {tag} Stage-2 ===')
        run_stage2(seq_dir, stage2_dir, max_frames=max_frames,
                   track_max_distance=track_max_distance)

    print(f'=== {tag} export CTC RES ===')
    if res_dir.exists():
        shutil.rmtree(res_dir)
    export_ctc_res(stage2_dir, res_dir)

    print(f'=== {tag} TRA ===')
    metrics = score_tra(gt_tra, man, res_dir, tag)
    row = {
        'dataset': dataset,
        'sequence': sequence,
        'backend': 'threshold',
        'track_max_distance': track_max_distance,
        'TRA': metrics.get('TRA'),
        'DET': metrics.get('DET'),
        'LNK': metrics.get('LNK'),
        'AOGM': metrics.get('AOGM'),
        'fn_nodes': metrics.get('fn_nodes'),
        'fp_nodes': metrics.get('fp_nodes'),
        'fn_edges': metrics.get('fn_edges'),
        'fp_edges': metrics.get('fp_edges'),
        'ns_nodes': metrics.get('ns_nodes'),
        'ws_edges': metrics.get('ws_edges'),
    }
    print(json.dumps(row, indent=2))
    out_json = BASE / 'scores' / f'{tag}_tra.json'
    out_json.parent.mkdir(parents=True, exist_ok=True)
    out_json.write_text(json.dumps({'summary': row, 'raw': metrics}, indent=2, default=str))
    return row

print('helpers ready')

## 2. Score one-by-one

After a code fix, re-run **setup cell** then the sequence cell.

If Stage-2 already finished for GOWT1_01, set `reuse_stage2=True` to skip re-segmentation.

In [ ]:
MAX_FRAMES = 0
TRACK_DIST = 50.0
rows = []

# --- 1/6 GOWT1 01 (reuse Stage-2 if still on disk) ---
rows.append(run_one(
    'Fluo-N2DH-GOWT1', '01',
    max_frames=MAX_FRAMES,
    track_max_distance=TRACK_DIST,
    reuse_stage2=True,  # set False to force full re-run
))

In [ ]:
rows.append(run_one('Fluo-N2DH-GOWT1', '02', max_frames=MAX_FRAMES, track_max_distance=TRACK_DIST))

In [ ]:
rows.append(run_one('Fluo-N2DH-SIM+', '01', max_frames=MAX_FRAMES, track_max_distance=TRACK_DIST))

In [ ]:
rows.append(run_one('Fluo-N2DH-SIM+', '02', max_frames=MAX_FRAMES, track_max_distance=TRACK_DIST))

In [ ]:
rows.append(run_one('Fluo-N2DL-HeLa', '01', max_frames=MAX_FRAMES, track_max_distance=TRACK_DIST))

In [ ]:
rows.append(run_one('Fluo-N2DL-HeLa', '02', max_frames=MAX_FRAMES, track_max_distance=TRACK_DIST))

## 3. Aggregate

In [ ]:
import pandas as pd
disk_rows = []
for p in sorted((BASE / 'scores').glob('*_tra.json')):
    disk_rows.append(json.loads(p.read_text())['summary'])
table = pd.DataFrame(disk_rows if disk_rows else rows)
display(table)
out_csv = BASE / 'scores' / 'tra_aggregate.csv'
table.to_csv(out_csv, index=False)
print('Wrote', out_csv)

### Notes
- TRA/DET/LNK from `traccuracy.CTCMetrics` — measured only.
- No division parents in OptiCell linker yet (P=0).
- Paste `tra_aggregate.csv` for the repo report.